# Plotting training results

This notebook serves as a visualization tool for the training results of the **FlowerLLM** training results.
It reads the metric from the training curves directly from Wandb and collect them in `pandas` data frames ready to be analyzed and plotted.

In [1]:
# Imports
from logging import INFO, ERROR
from typing import Any
import time
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.style as mplstyle
import matplotlib.patches as mpatches
import wandb
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MultipleLocator
import enum
from flwr.common import log

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "bold"
# Colors and line styles
color_palette = sns.color_palette("colorblind")
line_styles = ["-", "--", ":", "-."]
# Bolded axis labels
plt.rcParams["axes.labelweight"] = "bold"
mplstyle.use("fast")

In [2]:
def add_run_uuid(run_uuid: str) -> None:
    api = wandb.Api(timeout=10000)
    for i in range(64):
        try:
            actual_run_uuid = f"{run_uuid}_client_{i}"
            run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
            run.config["run_uuid"] = run_uuid
            run.update()
        except Exception as e:
            log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}_server"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}_centralised"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)


def change_run_uuid(old_run_uuid: str, new_run_uuid: str) -> None:
    api = wandb.Api(timeout=10000)
    for i in range(64):
        try:
            actual_run_uuid = f"{old_run_uuid}_client_{i}"
            run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
            run.config["run_uuid"] = new_run_uuid
            run.update()
        except Exception as e:
            log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}_server"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}_centralised"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)

In [3]:
class CentSuffix(str, enum.Enum):
    """Enum class for suffixes of centralized runs."""

    CENTRALIZED = "_centralised"
    CLIENT_0 = "_client_0"
    NONE = ""

In [4]:
class AxisLabels(str, enum.Enum):
    """Enum class for x- or y-axis labels."""

    FED_ROUND = "Federated Round"
    PPL = "Perplexity"

In [5]:
class PlottingCosmetics(str, enum.Enum):
    """Enum class for plotting cosmetics constants."""

    UPPER_RIGHT_POS = "upper right"
    UPPER_CENTER_POS = "upper center"

In [6]:
# Set the run_id to be retrieved
run_id_dict = {
    "full_participation": {
        # Number of local steps per round
        "64_steps": {
            "8": ("fed-lr-sched1-8cpr64-bs32-20241023_232107", 8, True, False),
            "4": (
                "fed-lr-sched0-4cpr64-bs32-20241022_145655",
                4,
                True,
                False,
            ),  # This has sched0 but it's actually sched1
            "2": ("fed-lr-sched1-2cpr64-bs32-20241023_144602", 2, True, False),
            "1": (
                "fed-lr-sched0-1cpr64-bs32-20241022_135427",
                1,
                True,
                False,
            ),  # This is of course repeated for several experiments as it is always the same
        },
        "128_steps": {
            "8": ("fed-steps-8cpr128-bs32-20241026_004019", 8, True, False),
            "4": ("fed-steps-4cpr128-bs32-20241025_185132", 4, True, False),
            "2": ("", 2, True, False),  # Running on CaMLSys cluster
            "1": ("fed-lr-sched0-1cpr64-bs32-20241022_135427", 1, True, False),
        },
        "512_steps": {
            "16": (
                "fed-steps-16cpr512-bs32-20241027_083734",
                16,
                True,
                False,
            ),  # Running on Lambda cluster
            "8": ("fed-steps-8cpr512-bs32-20241025_151104", 8, True, False),
            "4": ("fed-steps-4cpr512-bs32-20241025_110058", 4, True, False),
            "2": ("fed-steps-2cpr512-bs32-20241026_132229", 2, True, False),
            "1": ("fed-lr-sched0-1cpr64-bs32-20241022_135427", 1, True, False),
        },
    },
    "partial_participation": {
        # Participation ratios - 512 steps per round
        "0_5": {
            "8": ("fed-pp0_5-8cpr512-bs32-20241026_150027", 8, True, False),
            "4": ("fed-pp0_5-4cpr512-bs32-20241026_173006", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
        "0_25": {
            "8": ("", 8, True, False),
            "4": ("", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
        "0_125": {
            "8": ("fed-pp0_125-8cpr512-bs32-20241026_235833", 8, True, False),
            "4": ("fed-pp0_125-4cpr512-bs32-20241027_030224", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
    },
    "outer_optimizer": {
        # Types of outer optimizers - 512 steps per round
        # partial participation?
        "diloco": {
            "8": ("", 8, True, False),
            "4": ("", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
    },
    "non_iid_data": {
        # Heterogenous data sources - 512 steps per round (eval on C4 test set)
        # Full participation using 4 data sources each of which is split into 2 for the
        # 8cpr and 4 for the 16cpr.
        "the_pile": {
            "8": ("", 8, True, False),
            "4": ("", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
    },
    # ?
    "reset_opt_states": {},
}

In [7]:
# Server metrics columns
server_metrics_columns = [
    # Train-specific metrics
    "LanguageCrossEntropy",
    "LanguagePerplexity",
    # Val-specific metrics
    "ValLanguageCrossEntropy",
    "ValLanguagePerplexity",
    # Wandb internals
    "_runtime",
    "_step",
    "_timestamp",
    # Client-side bookkeeping
    "client/eval_init_time",
    "client/eval_metrics_collection_time",
    "client/eval_time",
    "client/eval_trainer_closing_time",
    "client/fit_get_parameters_time",
    "client/fit_init_time",
    "client/fit_metrics_collection_time",
    "client/fit_time",
    "client/fit_trainer_closing_time",
    # Client-side metrics
    "client/l2_norm_pseudo_gradient",
    "client_state_acc",
    "distributed_loss",
    # Node-side bookkeeping
    "node_eval_time_s",
    "node_training_time_s",
    # Server-side bookkeeping
    "server/evaluate_round_time",
    "server/evaluate_time",
    "server/first_check_nm_time",
    "server/fit_round_time",
    "server/round_time",
    "server/second_check_nm_time",
    # Server-side computed norms
    "server/l2_norm_fedavg_result",
    "server/l2_norm_model",
    "server/l2_norm_momentum_vector",
    "server/l2_norm_pseudo_gradient",
    # Step
    "step",
    # Worker-side bookkeeping
    "worker/partial_aggregation_time",
]

In [8]:
# Client metrics columns
client_metrics_columns = [
    # Wandb internals
    "_runtime",
    "_step",
    "_timestamp",
    # Activation norms
    "activations/l2_norm/full_model_input",
    "activations/l2_norm/full_model_output",
    # CID
    "client_id",
    # Momentum, gradient and model norms
    "l2_norm/grad/global",
    "l2_norm/moment/global",
    "l2_norm/param/global",
    "l2_norm/update/global",
    # Train loss
    "loss/train/total",
    # LR
    "lr-DecoupledAdamW/group0",
    # Memory metrics
    "memory/alloc_retries",
    "memory/current_active_mem",
    "memory/current_allocated_mem",
    "memory/current_inactive_mem",
    "memory/current_reserved_mem",
    "memory/peak_active_mem",
    "memory/peak_allocated_mem",
    "memory/peak_inactive_mem",
    "memory/peak_reserved_mem",
    "metrics/train/LanguageCrossEntropy",
    "metrics/train/LanguagePerplexity",
    # Step
    "step",
    # Throughput metrics
    "throughput/batches_per_sec",
    "throughput/device/batches_per_sec",
    "throughput/device/flops_per_sec",
    "throughput/device/mfu",
    "throughput/device/samples_per_sec",
    "throughput/device/tokens_per_sec",
    "throughput/flops_per_sec",
    "throughput/samples_per_sec",
    "throughput/tokens_per_sec",
    # Time metrics
    "time/batch",
    "time/batch_in_epoch",
    "time/epoch",
    "time/remaining_estimate",
    "time/sample",
    "time/sample_in_epoch",
    "time/token",
    "time/token_in_epoch",
    "time/total",
    "time/train",
    "time/val",
    # Microbatch size
    "trainer/device_train_microbatch_size",
]

In [9]:
x_lim: tuple[float, float] | None = None
perplexity_y_lim: dict[str, float] | None = {"top": 225, "bottom": 0}
norm_y_lim: dict[str, float] | None = {"bottom": 300, "top": 900}
momentum_y_lim: dict[str, float] | None = {"bottom": 300, "top": 900}
grad_y_lim: dict[str, float] | None = {"bottom": 0, "top": 125}
grad_client_y_lim: dict[str, float] | None = {"bottom": 0, "top": 1.6}
lr_y_lim: dict[str, float] | None = None
act_out_y_lim: dict[str, float] | None = None
act_out_y_lim: dict[str, float] | None = {"bottom": 0, "top": 1.3 * 1e7}
perplexity_legend_kwargs: dict[str, Any] | None = None
l2_gradient_legend_kwargs: dict[str, Any] | None = None

In [10]:
def download_metrics(
    run_id: str, n_clients: int, drop_layers: bool = True, use_server_name: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Download all the metrics from given run_id using Wandb API."""
    # Initialize the wandb API
    api = wandb.Api(timeout=100)
    # Get the run object
    server_run = (
        api.run(f"camlsys/pollen-llm/{run_id}_server")
        if use_server_name
        else api.run(f"camlsys/pollen-llm/{run_id}")
    )
    # Get the metrics
    server_metrics = server_run.scan_history()
    # Convert the metrics to a pandas data frame
    server_metrics_df = pd.DataFrame(server_metrics)
    # Drop the layers columns if needed
    if drop_layers:
        layer_columns = [col for col in server_metrics_df.columns if "layer" in col]
        server_metrics_df = server_metrics_df.drop(columns=layer_columns)
    # Add the `step` column based on the index
    server_metrics_df["step"] = server_metrics_df.index
    # Get clients metrics
    clients_metrics_df_list: list[pd.DataFrame] = []

    for i in range(n_clients):
        try:
            # Get the run object

            client_run = (
                api.run(f"camlsys/pollen-llm/{run_id}_client_{i}")
                if (
                    run_id
                    not in {
                        "pile-75M-final-20240605_133225",
                        "partial-75M-20240605_133326",
                    }
                )
                else api.run(f"camlsys/pollen-llm/{run_id}client_{i}")
            )
            # Get the metrics
            client_metrics = client_run.scan_history()
            # Convert the metrics to a pandas data frame
            client_metrics_df = pd.DataFrame(client_metrics)
            # Drop the layers columns if needed
            if drop_layers:
                layer_columns = [
                    col for col in client_metrics_df.columns if "layer" in col
                ]
                client_metrics_df = client_metrics_df.drop(columns=layer_columns)
            # Add the `step` column based on the index
            client_metrics_df["step"] = client_metrics_df._step
            # Add the `client_id` column based on the current client_id
            client_metrics_df["client_id"] = i
            # Append the client metrics to the list
            clients_metrics_df_list.append(client_metrics_df)
        except Exception as e:
            log(
                ERROR,
                f"Client {i} not found for run {run_id}",
                stack_info=True,
                exc_info=e,
            )
    # Concatenate the clients metrics
    clients_metrics_df = pd.concat(clients_metrics_df_list)
    # Return the metrics data frame
    return server_metrics_df, clients_metrics_df

In [11]:
# Wall time modeling
# Total Wall Time = (Number of Rounds * Round Completion Time)
# Round Completion Time (RCT) = (Client Training Time + Server Time)
# Client Training Time (CTT) = (Number of Local Steps / Local Throughput)
# Server Time - Parameter Server (PS) = (Broadcast Model Time + Collect Pseudo-gradients Time + Aggregation Time)
# Server Time - AllReduce (AR) = (AllReduce Communication Time + Aggregation Time)
# Server Time - RingAllReduce (RAR) = (RingAllReduce Communication Time + Aggregation Time)
# Broadcast Model Time = (Parameter Server Communication Complexity / Server Network Bandwidth)
# Collect Pseudo-gradients Time = (Parameter Server Communication Complexity / Server Network Bandwidth)
# Parameter Server Communication Complexity = (Number of Clients * Number of Parameters)
# Aggregation Time = (Aggregation Operations / Server FLOPs per second)
# Aggregation Operations = (Number of Clients * Number of Parameters)
# AllReduce Communication Time = (AllReduce Communication Complexity / Server Network Bandwidth)
# AllReduce Communication Complexity = ((Number of Clients - 1) * Number of Parameters)
# RingAllReduce Communication Time = (RingAllReduce Communication Complexity / Server Network Bandwidth)
# RingAllReduce Communication Complexity = (2 * Number of Parameters * (Number of Clients - 1) / Number of Clients)


class WallTimeModel:
    def __init__(
        self,
        num_rounds: int,
        num_clients_per_round: int,
        num_parameters: int,
        local_steps: int,
        local_throughput: float,  # Must be in batches per second
        server_bandwidth: float,  # Must be in MBps
        server_flops: float,  # Must be in FLOPs per second
        channels_threshold: int = 100,
    ):
        self.num_rounds = num_rounds
        self.num_clients_per_round = num_clients_per_round
        self.parameters_mbytes = num_parameters
        self.parameters_bytes = 2 * num_parameters
        self.parameters_mbytes = self.parameters_bytes / 10**6
        self.local_steps = local_steps
        self.local_throughput = local_throughput
        self.server_bandwidth = server_bandwidth
        self.server_flops = server_flops
        self.channels_threshold = channels_threshold

    def client_training_time(self):
        return self.local_steps / self.local_throughput

    def broadcast_model_time(self):
        if self.num_clients_per_round > self.channels_threshold:
            return (self.num_clients_per_round * self.parameters_mbytes) / np.sqrt(
                self.server_bandwidth
            )
        else:
            return (
                self.num_clients_per_round * self.parameters_mbytes
            ) / self.server_bandwidth

    def collect_pseudo_gradients_time(self):
        if self.num_clients_per_round > self.channels_threshold:
            return (self.num_clients_per_round * self.parameters_mbytes) / np.sqrt(
                self.server_bandwidth
            )
        else:
            return (
                self.num_clients_per_round * self.parameters_mbytes
            ) / self.server_bandwidth

    def aggregation_time(self):
        return (self.num_clients_per_round * self.parameters_mbytes) / self.server_flops

    def allreduce_communication_time(self):
        return (
            (self.num_clients_per_round - 1) * self.parameters_mbytes
        ) / self.server_bandwidth

    def ring_allreduce_communication_time(self):
        return (
            2
            * self.parameters_mbytes
            * (self.num_clients_per_round - 1)
            / self.num_clients_per_round
        ) / self.server_bandwidth

    def parameter_server_time(self):
        return (
            self.broadcast_model_time()
            + self.collect_pseudo_gradients_time()
            + self.aggregation_time()
        )

    def allreduce_server_time(self):
        return self.allreduce_communication_time() + self.aggregation_time()

    def ring_allreduce_server_time(self):
        return self.ring_allreduce_communication_time() + self.aggregation_time()

    def round_completion_time_ps(self):
        return self.client_training_time() + self.parameter_server_time()

    def round_completion_time_ar(self):
        return self.client_training_time() + self.allreduce_server_time()

    def round_completion_time_rar(self):
        return self.client_training_time() + self.ring_allreduce_server_time()

    def total_wall_time_ps(self):
        return self.num_rounds * self.round_completion_time_ps()

    def total_wall_time_ar(self):
        return self.num_rounds * self.round_completion_time_ar()

    def total_wall_time_rar(self):
        return self.num_rounds * self.round_completion_time_rar()

In [12]:
# Initialize the model
model = WallTimeModel(
    num_rounds=100,
    num_clients_per_round=10,
    num_parameters=1_000_000,
    local_steps=1_000,
    local_throughput=100,
    server_bandwidth=1_000,
    server_flops=5_000_000_000,
)

# Calculate total wall time for different approaches
total_wall_time_ps = model.total_wall_time_ps()
total_wall_time_ar = model.total_wall_time_ar()
total_wall_time_rar = model.total_wall_time_rar()

print(f"Total Wall Time (Parameter Server): {total_wall_time_ps}")
print(f"Total Wall Time (AllReduce): {total_wall_time_ar}")
print(f"Total Wall Time (RingAllReduce): {total_wall_time_rar}")

Total Wall Time (Parameter Server): 1004.0000004
Total Wall Time (AllReduce): 1001.8000003999999
Total Wall Time (RingAllReduce): 1000.3600004000001


In [13]:
def find_federated_round_for_perplexity(
    client_metrics_df: pd.DataFrame,
    perplexity_value: float | None,
    rolling_window: int = 100,
) -> tuple[int, float]:
    """
    Find the federated round for a given perplexity value.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the metrics with columns 'steps' and 'metrics/train/LanguagePerplexity'.
    perplexity_value : float
        The perplexity value to find the federated round for.
    rolling_window : int, optional
        The window size for smoothing the perplexity series, by default 100.

    Returns
    -------
    int
        The federated round corresponding to the given perplexity value.
    """
    # Get columns
    exclude_columns = client_metrics_df.columns
    # Remove columns of interest
    exclude_columns = [
        col
        for col in exclude_columns
        if col not in {"step", "metrics/train/LanguagePerplexity"}
    ]
    aggregated_df = client_metrics_df.drop(columns=exclude_columns)
    # Aggregate by step
    aggregated_df = aggregated_df.groupby("step").mean().reset_index()
    # Extract `metrics/train/LanguagePerplexity` series against steps
    steps: pd.Series[int] = aggregated_df["step"]
    perplexity_series = aggregated_df["metrics/train/LanguagePerplexity"]

    # Smooth the line using a rolling window of 100 steps
    smoothed_perplexity = perplexity_series.rolling(window=rolling_window).mean()
    # log(INFO, f"Smoothed perplexity: {smoothed_perplexity}")

    # Find the x-coordinate of a given perplexity value
    if perplexity_value is None:
        perplexity_value = smoothed_perplexity.min()
    assert perplexity_value is not None
    closest_index = (smoothed_perplexity - perplexity_value).abs().idxmin()
    x_coordinate = steps[closest_index]
    log(INFO, f"Closest index: {closest_index}, x-coordinate: {x_coordinate}")

    # Return the step and the perplexity value
    return x_coordinate, perplexity_value

In [23]:
# Load the WandB data
experiment_type = "full_participation"
experiment_hp = "64_steps"
# conf = "4"
# TODO: We must decide on a fair PPL value and an harsh PPL value
min_ppl = 35  # Cherry-picked value
min_ppl = 41  # Centralized baseline value rounded down
min_ppl = 42  # Centralized baseline value rounded up
min_ppl = None  # Get the min from 1 client per round experiment
min_ppl = 41.1  # Centralized baseline value
min_ppl = 36  # Cherry picked
for conf in sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]]):
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    try:
        server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
        client_metrics_df=client_metrics_df, perplexity_value=min_ppl, rolling_window=10
    )
    log(INFO, f"Min PPL for conf {conf}: {new_min_ppl} at step {steps_to_min}")
    min_ppl = min(min_ppl, new_min_ppl) if min_ppl is not None else new_min_ppl
    # Initialize the model
    n_local_steps = int(experiment_hp.replace("_steps", ""))
    model = WallTimeModel(
        num_rounds=steps_to_min // n_local_steps,
        num_clients_per_round=int(conf),
        num_parameters=125_000_000,
        local_steps=n_local_steps,
        local_throughput=100,
        # server_bandwidth=1_000, # 1 Gbps
        # server_bandwidth=2_500,  # 2.5 Gbps
        server_bandwidth=5_000,  # 5 Gbps
        server_flops=5_000_000_000_000,  # 5 TFLOPs
    )

    # Calculate total wall time for different approaches
    total_wall_time_ps = model.total_wall_time_ps()
    total_wall_time_ar = model.total_wall_time_ar()
    total_wall_time_rar = model.total_wall_time_rar()

    log(INFO, "Total Wall Time for %s (Parameter Server): %s", conf, total_wall_time_ps)
    log(INFO, "Total Wall Time for %s (AllReduce): %s", conf, total_wall_time_ar)
    log(INFO, "Total Wall Time for %s (RingAllReduce): %s", conf, total_wall_time_rar)

INFO :      Closest index: 24669, x-coordinate: 24980
INFO :      Min PPL for conf 1: 36 at step 24980
INFO :      Total Wall Time for 1 (Parameter Server): 288.6000000195
INFO :      Total Wall Time for 1 (AllReduce): 249.60000001950002
INFO :      Total Wall Time for 1 (RingAllReduce): 249.60000001950002
INFO :      Closest index: 16444, x-coordinate: 17196
INFO :      Min PPL for conf 2: 36 at step 17196
INFO :      Total Wall Time for 2 (Parameter Server): 225.12000002680003
INFO :      Total Wall Time for 2 (AllReduce): 184.9200000268
INFO :      Total Wall Time for 2 (RingAllReduce): 184.9200000268
INFO :      Closest index: 14609, x-coordinate: 14677
INFO :      Min PPL for conf 4: 36 at step 14677
INFO :      Total Wall Time for 4 (Parameter Server): 238.16000004580002
INFO :      Total Wall Time for 4 (AllReduce): 180.91000004580002
INFO :      Total Wall Time for 4 (RingAllReduce): 163.7350000458
INFO :      Closest index: 15020, x-coordinate: 15020
INFO :      Min PPL for co